# 5.2 Densidad de Corriente y Ley de Ohm

Volviendo a nuestra expresión de corriente e incorporando la velocidad de deriva, obtenemos:

$$I = n \, A \, v_d \, q$$

Definimos la **densidad de corriente** ($\vec{J}$) como la corriente por unidad de área, un vector que apunta en la dirección del flujo de cargas positivas:

$$\vec{J} = \frac{I}{A}\hat{n} = n \, q \, \vec{v}_d$$

De forma general, la corriente macroscópica es el flujo de la densidad de corriente a través de una superficie:

$$I = \int \vec{J} \cdot d\vec{A}$$

## Ley de Ohm microscópica

Sabiendo que $v_d$ es proporcional al campo eléctrico, podemos establecer que la densidad de corriente es directamente proporcional al campo eléctrico aplicado. Esta relación es conocida como la Ley de Ohm microscópica:

$$\vec{J} = \sigma \vec{E} \quad \text{o bien} \quad \vec{J} = \frac{1}{\rho}\vec{E}$$

Donde $\sigma$ es la conductividad del material y $\rho$ es su resistividad ($\rho = 1/\sigma$).


In [4]:
from IPython.display import display, HTML

simulacion_ohm_mic_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Ley de Ohm Microscópica</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Observa un conductor con un estrechamiento central. En estado estacionario, la corriente total (I) es la misma en todas las secciones, pero el área transversal cambia.</p>
    
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px;">
        <div style="flex: 1.5; min-width: 300px;">
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Corriente Total (I): <span id="ohm_val_I" style="font-weight: normal;">50</span></label>
                <input type="range" id="ohm_slider_I" min="10" max="100" step="5" value="50" style="width: 35%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Conductividad (σ): <span id="ohm_val_Sigma" style="font-weight: normal;">Alta</span></label>
                <input type="range" id="ohm_slider_Sigma" min="1" max="5" step="1" value="4" style="width: 35%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Estrechamiento (Grosor): <span id="ohm_val_A2" style="font-weight: normal;">50%</span></label>
                <input type="range" id="ohm_slider_A2" min="20" max="100" step="10" value="50" style="width: 35%; vertical-align: middle;">
            </div>
        </div>

        <div style="flex: 1; min-width: 250px; font-size: 0.9em; background: #fff; padding: 10px; border-radius: 6px; border: 1px solid #ccc;">
            <div style="margin-bottom: 5px;"><u>Zona Ancha (1)</u> vs <u>Zona Estrecha (2)</u></div>
            <div style="margin-bottom: 5px;"><b>Densidad (J):</b> <span id="ohm_val_J1" style="color: #0d6efd;">1.0</span> | <span id="ohm_val_J2" style="color: #dc3545;">2.0</span> au</div>
            <div style="margin-bottom: 5px;"><b>Campo (E):</b> <span id="ohm_val_E1" style="color: #0d6efd;">0.25</span> | <span id="ohm_val_E2" style="color: #dc3545;">0.50</span> au</div>
            <div style="font-size: 0.8em; color: #777; margin-top: 5px;">* J = I / A <br>* E = J / σ (Ley de Ohm Microscópica)</div>
        </div>
    </div>
    
    <div style="text-align: center; position: relative;">
        <canvas id="ohm_canvas" width="600" height="250" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    setTimeout(function() {
        const canvas = document.getElementById('ohm_canvas');
        if (!canvas) return;
        const ctx = canvas.getContext('2d');
        
        const slider_I = document.getElementById('ohm_slider_I');
        const slider_Sigma = document.getElementById('ohm_slider_Sigma');
        const slider_A2 = document.getElementById('ohm_slider_A2');
        
        // Elementos UI
        const val_I = document.getElementById('ohm_val_I');
        const val_Sigma = document.getElementById('ohm_val_Sigma');
        const val_A2 = document.getElementById('ohm_val_A2');
        const val_J1 = document.getElementById('ohm_val_J1');
        const val_J2 = document.getElementById('ohm_val_J2');
        const val_E1 = document.getElementById('ohm_val_E1');
        const val_E2 = document.getElementById('ohm_val_E2');
        
        let electrons = [];
        const num_electrons = 200;
        
        function initElectrons() {
            electrons = [];
            for(let i=0; i<num_electrons; i++) {
                electrons.push({
                    x: Math.random() * canvas.width,
                    y: Math.random() // Posición relativa en Y (0 a 1) para escalar con el tubo
                });
            }
        }

        function drawArrow(x, y, length, color, label) {
            ctx.beginPath();
            ctx.moveTo(x, y);
            ctx.lineTo(x + length, y);
            ctx.strokeStyle = color;
            ctx.lineWidth = 2;
            ctx.stroke();
            
            // Punta de flecha
            ctx.beginPath();
            ctx.moveTo(x + length, y);
            ctx.lineTo(x + length - 6, y - 4);
            ctx.lineTo(x + length - 6, y + 4);
            ctx.fillStyle = color;
            ctx.fill();
            
            if(label) {
                ctx.fillStyle = color;
                ctx.font = "bold 12px Arial";
                ctx.fillText(label, x + length/2 - 5, y - 8);
            }
        }

        function animate() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            let I = parseFloat(slider_I.value);
            let sigma = parseFloat(slider_Sigma.value);
            let A2_percent = parseFloat(slider_A2.value) / 100;
            
            val_I.innerText = I;
            val_A2.innerText = slider_A2.value + "%";
            let sigma_labels = ["Muy Baja", "Baja", "Media", "Alta", "Muy Alta"];
            val_Sigma.innerText = sigma_labels[sigma - 1];
            
            // Geometría del conductor
            let H1 = 160; // Altura zona ancha
            let H2 = H1 * A2_percent; // Altura zona estrecha
            let cy = canvas.height / 2;
            
            let x_start = 150;
            let x_end = 450;
            let transition = 40;
            
            // Dibujar Conductor (Fondo)
            ctx.fillStyle = "#e9ecef";
            ctx.beginPath();
            ctx.moveTo(0, cy - H1/2);
            ctx.lineTo(x_start, cy - H1/2);
            ctx.lineTo(x_start + transition, cy - H2/2);
            ctx.lineTo(x_end - transition, cy - H2/2);
            ctx.lineTo(x_end, cy - H1/2);
            ctx.lineTo(canvas.width, cy - H1/2);
            
            ctx.lineTo(canvas.width, cy + H1/2);
            ctx.lineTo(x_end, cy + H1/2);
            ctx.lineTo(x_end - transition, cy + H2/2);
            ctx.lineTo(x_start + transition, cy + H2/2);
            ctx.lineTo(x_start, cy + H1/2);
            ctx.lineTo(0, cy + H1/2);
            ctx.closePath();
            ctx.fill();
            ctx.strokeStyle = "#adb5bd";
            ctx.lineWidth = 2;
            ctx.stroke();

            // Cálculos Físicos
            let J1 = I / H1;
            let J2 = I / H2;
            let E1 = J1 / sigma;
            let E2 = J2 / sigma;
            
            val_J1.innerText = (J1 * 10).toFixed(2);
            val_J2.innerText = (J2 * 10).toFixed(2);
            val_E1.innerText = (E1 * 100).toFixed(2);
            val_E2.innerText = (E2 * 100).toFixed(2);

            // Mover y dibujar electrones
            ctx.fillStyle = "#0d6efd";
            electrons.forEach(e => {
                let current_H = H1;
                let current_J = J1;
                
                // Determinar en qué zona está la partícula
                if (e.x > x_start + transition && e.x < x_end - transition) {
                    current_H = H2;
                    current_J = J2;
                } else if (e.x >= x_start && e.x <= x_start + transition) {
                    // Interpolación lineal simple para la transición
                    let ratio = (e.x - x_start) / transition;
                    current_H = H1 - (H1 - H2) * ratio;
                    current_J = I / current_H;
                } else if (e.x >= x_end - transition && e.x <= x_end) {
                    let ratio = (e.x - (x_end - transition)) / transition;
                    current_H = H2 + (H1 - H2) * ratio;
                    current_J = I / current_H;
                }
                
                // La velocidad de deriva es proporcional a J
                let vd = current_J * 2; 
                e.x += vd;
                
                if (e.x > canvas.width) e.x = 0;
                
                // Calcular la posición Y real basada en la altura actual del tubo
                let real_y = cy - (current_H/2) + (e.y * current_H);
                
                ctx.beginPath();
                ctx.arc(e.x, real_y, 2.5, 0, Math.PI * 2);
                ctx.fill();
            });
            
            // Dibujar vectores macroscópicos representativos
            drawArrow(50, cy - 30, J1 * 50, "#0d6efd", "J");
            drawArrow(50, cy + 30, E1 * 500, "#dc3545", "E");
            
            drawArrow(300, cy - 15, J2 * 50, "#0d6efd", "J");
            drawArrow(300, cy + 15, E2 * 500, "#dc3545", "E");
            
            requestAnimationFrame(animate);
        }

        slider_I.addEventListener('input', () => {});
        slider_Sigma.addEventListener('input', () => {});
        slider_A2.addEventListener('input', () => {});

        initElectrons();
        animate();
    }, 150);
</script>
"""
display(HTML(simulacion_ohm_mic_html))


## Ley de Ohm macroscópica

Si aplicamos la relación microscópica a un segmento de cable de longitud $L$ y área transversal $A$, la diferencia de potencial $\Delta V$ entre los extremos es:

$$\Delta V = E L$$

Sustituyendo el campo eléctrico $E = \rho J$ y sabiendo que $J = I/A$:

$$\Delta V = \left(\rho \frac{I}{A}\right) L = \left(\frac{\rho L}{A}\right) I$$

El término entre paréntesis agrupa las propiedades geométricas del material, definiendo la **Resistencia** ($R$):

$$R = \frac{\rho L}{A}$$

Llegando así a la conocida forma macroscópica de la Ley de Ohm:

$$\Delta V = R I$$

> **Nota:** La unidad de la resistencia es el Ohmio, $1 \, \Omega = 1 \text{ V/A}$



## Dependencia de la temperatura

La resistividad de un material no es estrictamente constante, sino que depende de la temperatura. Para rangos moderados, obedece a una relación lineal:

$$\rho(T) = \rho_0(1 + \alpha(T - T_0))$$

Donde $\alpha$ es el coeficiente térmico del material. Consecuentemente, la resistencia también varía:

$$R(T) = R_0(1 + \alpha(T - T_0))$$

En algunos materiales, al descender la temperatura por debajo de un valor crítico $T_c$, la resistividad cae abruptamente a cero, fenómeno conocido como **superconductividad**.

## Ejemplo: Esferas conductoras concéntricas

Consideremos dos esferas conductoras concéntricas de radios $R_1$ (interior) y $R_2$ (exterior) con un material de resistividad $\rho$ entre ellas. 
Si fluye una corriente $I$ con simetría radial, la densidad de corriente es:

$$\vec{J} = \frac{I}{4\pi r^2}\hat{r}$$

Por la ley de Ohm microscópica, el campo eléctrico en la región intermedia es:

$$E = \rho J = \frac{\rho I}{4\pi r^2}$$

La diferencia de potencial $\Delta V$ entre las esferas se calcula integrando el campo eléctrico:

$$\Delta V = -\int_{R_2}^{R_1} \vec{E} \cdot d\vec{r} = -\int_{R_2}^{R_1} \frac{\rho I}{4\pi r^2} dr = \frac{\rho I}{4\pi} \left[ \frac{1}{r} \right]_{R_2}^{R_1}$$

$$\Delta V = \frac{\rho I}{4\pi} \left( \frac{1}{R_1} - \frac{1}{R_2} \right) = \frac{\rho I}{4\pi} \left( \frac{R_2 - R_1}{R_1 R_2} \right)$$

Dado que macroscópicamente $\Delta V = I R$, podemos identificar la resistencia equivalente de este arreglo:

$$R = \frac{\rho}{4\pi} \left( \frac{R_2 - R_1}{R_1 R_2} \right)$$



## Preguntas de Análisis


In [1]:
from IPython.display import display, HTML

quiz_densidad_html = """
<style>
    .fisi-quiz-container {
        border: 1px solid #e0e0e0;
        padding: 15px;
        margin-bottom: 25px;
        border-radius: 8px;
        background-color: #f8f9fa;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    }
    .fisi-question {
        font-weight: 600;
        margin-bottom: 15px;
        color: #2c3e50;
    }
    .fisi-option {
        margin-bottom: 10px;
        display: flex;
        align-items: flex-start;
        cursor: pointer;
    }
    .fisi-option input {
        margin-top: 4px;
        margin-right: 10px;
    }
    .fisi-check-btn {
        background-color: #f73bc5;
        color: white;
        border: none;
        padding: 8px 20px;
        border-radius: 4px;
        cursor: pointer;
        font-size: 14px;
        margin-top: 10px;
        font-weight: 500;
    }
    .fisi-check-btn:hover { background-color: #ab2988; }
    .fisi-feedback {
        margin-top: 15px;
        padding: 12px;
        border-radius: 6px;
        display: none;
        font-size: 0.95em;
    }
</style>

<script>
    if (typeof window.checkFisiQuiz === 'undefined') {
        window.checkFisiQuiz = function(radioName, feedbackId) {
            var radios = document.getElementsByName(radioName);
            var selected = null;
            var feedbackDiv = document.getElementById(feedbackId);
            
            for (var i = 0; i < radios.length; i++) {
                if (radios[i].checked) {
                    selected = radios[i];
                    break;
                }
            }
            
            feedbackDiv.style.display = 'block';
            
            if (!selected) {
                feedbackDiv.style.backgroundColor = '#fff3cd';
                feedbackDiv.style.color = '#856404';
                feedbackDiv.style.border = '1px solid #ffeeba';
                feedbackDiv.innerHTML = '⚠️ Por favor, selecciona una opción.';
                return;
            }
            
            var isCorrect = selected.getAttribute('data-correct') === 'true';
            var msg = selected.getAttribute('data-fb');
            
            if (isCorrect) {
                feedbackDiv.style.backgroundColor = '#d4edda';
                feedbackDiv.style.color = '#155724';
                feedbackDiv.style.border = '1px solid #c3e6cb';
                feedbackDiv.innerHTML = '✅ <strong>¡Correcto!</strong><br>' + msg;
            } else {
                feedbackDiv.style.backgroundColor = '#f8d7da';
                feedbackDiv.style.color = '#721c24';
                feedbackDiv.style.border = '1px solid #f5c6cb';
                feedbackDiv.innerHTML = '❌ <strong>Incorrecto.</strong><br>' + msg;
            }
        };
    }
</script>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        1. Teniendo en cuenta la simulación y la teoría sobre un conductor con un estrechamiento (donde el área transversal A disminuye a la mitad), ¿qué ocurre de forma simultánea con la corriente total (I) y la densidad de corriente (J) al pasar por la zona estrecha en estado estacionario?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qden1" value="a" data-fb="Incorrecto. Si la corriente se redujera a la mitad, se acumularían cargas en el embudo, rompiendo el estado estacionario."> 
        <span>Ambas, la corriente total y la densidad de corriente, se reducen a la mitad como consecuencia geométrica.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden1" value="b" data-correct="true" data-fb="¡Exacto! Por conservación de la carga en estado estacionario, los mismos electrones que entran deben salir; por lo tanto, la corriente I se mantiene constante en todo el circuito. Sin embargo, al pasar la misma corriente por la mitad del área, la densidad de corriente J = I/A inevitablemente se duplica, aumentando localmente el campo eléctrico y la velocidad de deriva."> 
        <span>La corriente total I se mantiene constante en todo el conductor, pero la densidad de corriente J se duplica en la zona estrecha.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden1" value="c" data-fb="Incorrecto. La corriente no se crea; lo que entra por un lado debe salir por el otro, por lo que la corriente no se duplica."> 
        <span>La corriente total I se duplica en la zona del estrechamiento, mientras que la densidad de corriente J permanece inalterada.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden1" value="d" data-fb="Incorrecto. Si divides la misma corriente entre un área más pequeña, el resultado (la densidad) es mayor, no menor."> 
        <span>La densidad de corriente J se reduce a la mitad para compensar la falta de espacio y así mantener estable la corriente total I.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qden1', 'feedback-den1')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-den1"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        2. La Ley de Ohm microscópica se expresa vectorialmente como J = σE. Según esta relación matemática, si analizamos un material que posee una conductividad (σ) extremadamente alta, ¿qué podemos deducir acerca del campo eléctrico interno (E) necesario para mantener una cierta densidad de corriente (J) constante?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qden2" value="a" data-fb="Incorrecto. Si el material es un excelente conductor (alta conductividad), es muy fácil mover las cargas, por lo que requerirás menos fuerza, no más."> 
        <span>Se requiere un campo eléctrico extremadamente grande para poder impulsar las cargas a través de la red cristalina.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden2" value="b" data-correct="true" data-fb="¡Correcto! Si despejamos el campo obtenemos E = J/σ. Si σ es muy grande, el cociente se vuelve diminuto. Esto significa que en un material muy buen conductor, se necesita un esfuerzo (campo E o diferencia de potencial) casi despreciable para lograr que fluya una gran cantidad de densidad de corriente."> 
        <span>El campo eléctrico necesario será muy pequeño, aproximándose a cero en el caso límite de un conductor ideal.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden2" value="c" data-fb="Incorrecto. La alta conductividad es un escalar positivo; no invierte el signo del vector ni la dirección del flujo de corriente."> 
        <span>El campo eléctrico se vuelve negativo, invirtiendo la dirección natural del vector de densidad de corriente.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden2" value="d" data-fb="Incorrecto. La conductividad sí afecta directamente la relación entre J y E en la forma microscópica de la Ley de Ohm."> 
        <span>La conductividad es irrelevante para el campo eléctrico interno; este solo depende de la longitud (L) del material.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qden2', 'feedback-den2')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-den2"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        3. Partiendo de la definición puramente geométrica de la resistencia macroscópica (R = ρ·L / A), si quisieras fabricar un cable de cobre que tenga la menor resistencia eléctrica posible para reducir el Efecto Joule, ¿qué proporciones deberías priorizar?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qden3" value="a" data-fb="Incorrecto. Un cable largo (aumenta L) y delgado (disminuye A) haría que ambos factores en la fórmula aumenten la resistencia final del cable."> 
        <span>Debe ser un cable sumamente largo y muy delgado, para que la corriente viaje más rápido por efecto embudo.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden3" value="b" data-fb="Incorrecto. La resistividad (ρ) depende de la temperatura y del material, no del grosor o largo geométrico del cable."> 
        <span>Debe tener una longitud y grosor intermedios, logrando que la resistividad interna (ρ) del cobre disminuya por disipación térmica.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden3" value="c" data-correct="true" data-fb="¡Exacto! Según R = ρ·L / A, para minimizar la resistencia 'R' necesitas que el numerador (la longitud L) sea lo más pequeño posible, y el denominador (el área transversal A) sea lo más grande posible. Físicamente, un camino corto y un 'túnel' ancho facilitan enormemente el flujo de electrones."> 
        <span>Debe ser un cable lo más corto posible (menor L) y lo más grueso posible (mayor área transversal A).</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden3" value="d" data-fb="Incorrecto. El enrollar el cable no cancela la densidad de corriente. De hecho, incrementaría la longitud total L utilizada, subiendo la resistencia."> 
        <span>Debe ser un alambre retorcido en forma de espiral helicoidal para cancelar las componentes de la densidad de corriente.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qden3', 'feedback-den3')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-den3"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        4. Sabemos que en condiciones normales, la resistividad térmica de un metal aumenta de forma lineal al incrementar su temperatura. Según se menciona en la sección, ¿qué ocurre microscópicamente en ciertos materiales si la temperatura desciende por debajo de un valor crítico conocido como Tc?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qden4" value="a" data-fb="Incorrecto. Disminuir la temperatura reduce la vibración de la red, facilitando el paso. No hace que la resistividad aumente."> 
        <span>Su resistividad aumenta de forma asintótica debido a que los electrones de conducción quedan "congelados" en sus órbitas.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden4" value="b" data-fb="Incorrecto. Un aislante perfecto sería un material con resistencia infinita (conductividad cero), lo que es el efecto contrario al descrito."> 
        <span>Su conductividad cae a cero absoluto, convirtiendo al material temporalmente en un aislante eléctrico perfecto o dieléctrico.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden4" value="c" data-correct="true" data-fb="¡Correcto! En el fenómeno de la superconductividad, al bajar de la temperatura crítica (Tc), la resistividad no solo disminuye, sino que desaparece por completo (ρ = 0). Esto permite que las corrientes eléctricas circulen indefinidamente sin pérdida de energía por Efecto Joule."> 
        <span>Su resistividad cae abruptamente a exactamente cero, permitiendo el flujo ininterrumpido de corriente sin disipación de energía (superconductividad).</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden4" value="d" data-fb="Incorrecto. La resistividad es una propiedad escalar intrínseca; no se 'invierte' forzando a la corriente a desobedecer la fuente de voltaje."> 
        <span>Su resistividad se invierte hacia valores negativos, provocando que la corriente fluya espontáneamente en sentido contrario al campo aplicado.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qden4', 'feedback-den4')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-den4"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        5. En el análisis matemático de la resistencia entre dos esferas conductoras concéntricas sometidas a una corriente I con simetría radial, se deduce que el campo eléctrico E decae de forma proporcional a 1/r². ¿Cuál es la razón geométrica y física que obliga a este decaimiento radial?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qden5" value="a" data-fb="Incorrecto. Por el principio de conservación, la corriente total I en estado estacionario es la misma sin importar qué radio cruce."> 
        <span>Porque la corriente macroscópica total I va consumiéndose y disminuyendo gradualmente a medida que avanza desde la esfera interior a la exterior.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden5" value="b" data-correct="true" data-fb="¡Exacto! La corriente I debe atravesar superficies esféricas imaginarias que crecen mientras nos alejamos. El área de estas superficies es A = 4πr². Al calcular la densidad J = I / (4πr²), la densidad disminuye como 1/r², y como la Ley de Ohm microscópica afirma que E = ρJ, el campo eléctrico decae con esa misma proporción espacial."> 
        <span>Porque la densidad de corriente (J) está inversamente relacionada al área de la superficie esférica de paso (4πr²), la cual aumenta al alejarnos del centro.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden5" value="c" data-fb="Incorrecto. La resistividad (ρ) es una propiedad del material (constante en este caso uniforme); no depende espacialmente del radio."> 
        <span>Porque la resistividad interna (ρ) del material conductor sólido entre las esferas aumenta proporcionalmente al cuadrado del radio.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qden5" value="d" data-fb="Incorrecto. La Ley de Ohm microscópica (J = σE) es universal a nivel local y se aplica independientemente de la geometría global del problema."> 
        <span>Porque la Ley de Ohm microscópica es únicamente válida para cables cilíndricos rectos, presentando un error predictivo en simetrías esféricas.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qden5', 'feedback-den5')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-den5"></div>
</div>
"""

display(HTML(quiz_densidad_html))